# 🎯 Notebook 03: Fine-tuning LLM với QLoRA

Notebook này thực hiện:
1. Load Q&A dataset
2. Fine-tune Qwen2.5-7B (hoặc Vistral/PhoGPT)
3. Save LoRA adapters

⚠️ Yêu cầu: GPU A100 40GB (Google Colab Pro/Pro+)

In [1]:
import os, sys
PROJECT_DIR = '/content/vietnamese-legal-qa'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

from src.config.settings import Settings
from src.services.fine_tuning_service import FineTuningService

settings = Settings.load('config.yaml')
ft_service = FineTuningService(settings)

FileNotFoundError: [Errno 2] No such file or directory: '/content/vietnamese-legal-qa'

In [ ]:
# === Load Q&A dataset ===
qa_pairs = ft_service.load_dataset('data/alqac')
print(f'Loaded {len(qa_pairs)} Q&A pairs for fine-tuning')

# Preview format
from src.components.fine_tuner import FineTuner
ft = FineTuner(settings)
sample = ft._format_chatml(qa_pairs[0])
print(f'\nSample formatted input:\n{sample[:500]}')

In [ ]:
# === Fine-tune Qwen2.5-7B ===
# Đây là model khuyến nghị (hỗ trợ tiếng Việt tốt, ổn định)
result_qwen = ft_service.finetune_model(
    model_key='qwen',
    qa_pairs=qa_pairs,
    format_type='chatml'
)
print(f'\n✅ Qwen2.5 fine-tuning complete!')
print(f'   Loss: {result_qwen.final_loss:.4f}')
print(f'   Time: {result_qwen.training_time_minutes:.1f} min')
print(f'   Adapter: {result_qwen.adapter_path}')

In [ ]:
# === Fine-tune Vistral-7B ===
# ⚠️ Chạy riêng (cần restart runtime để free VRAM)
result_vistral = ft_service.finetune_model(
    model_key='vistral',
    qa_pairs=qa_pairs,
    format_type='chatml'
)
print(f'\n✅ Vistral fine-tuning complete!')
print(f'   Loss: {result_vistral.final_loss:.4f}')

In [ ]:
# === Fine-tune PhoGPT-7.5B ===
# ⚠️ Chạy riêng (cần restart runtime để free VRAM)
result_phogpt = ft_service.finetune_model(
    model_key='phogpt',
    qa_pairs=qa_pairs,
    format_type='chatml'
)
print(f'\n✅ PhoGPT fine-tuning complete!')
print(f'   Loss: {result_phogpt.final_loss:.4f}')

In [ ]:
# === Copy adapters to Google Drive (backup) ===
import shutil
drive_backup = '/content/drive/MyDrive/legal-qa-adapters/'
os.makedirs(drive_backup, exist_ok=True)

for model_key in ['qwen', 'vistral', 'phogpt']:
    src = f'models/adapters/{model_key}'
    if os.path.exists(src):
        dst = os.path.join(drive_backup, model_key)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Backed up {model_key} adapter to Drive')

print('\n✅ All adapters backed up!')